# Sprint Block A+B — Setup Harness + Wikivoyage / OSM / Commons Collection

**Runtime: CPU** (do NOT burn GPU quota on this — save it for annotation & DistilBERT).

This notebook implements Hours 0–6 of the 48-hour runbook:
- **Block A**: Drive folders, checkpoint harness, pinned installs
- **Block B1**: Wikivoyage dump → anchored sentences (`{{see}}/{{do}}/{{eat}}/{{drink}}/{{sleep}}/{{listing}}` templates)
- **B2 / B3**: OSM PBF descriptions + gazetteer, Commons geosearch captions

Everything checkpoints to Drive every ~500 records. A disconnect costs you minutes, not hours. Re-running any cell resumes where it left off — every stage is idempotent.

**Parallelism tip:** open a second copy of this notebook in another tab, run Block A there too, then run B1 in this tab and B2+B3 in the other simultaneously.

## Block A — Cell 1: Mount Drive + folder structure

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT = '/content/drive/MyDrive/geoproject'
SUBDIRS = ['raw', 'clean', 'annotated', 'transactions', 'models',
           'models_cache', 'logs', 'checkpoints']
for d in SUBDIRS:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)

# Model/dataset caches live on Drive so we never re-download across sessions
os.environ['HF_HOME'] = f'{ROOT}/models_cache/hf'
os.environ['XDG_CACHE_HOME'] = f'{ROOT}/models_cache/xdg'
print('Drive mounted, folders ready at', ROOT)

## Block A — Cell 2: Pinned installs (~2 min)

In [ ]:
%%capture piplog
!pip install -q "mwparserfromhell>=0.6" "datasketch>=1.6" "rapidfuzz>=3.0" \
               "osmium>=3.6" "pandas>=2.0" "pyarrow>=14" "requests" "tqdm"


In [ ]:
import mwparserfromhell, osmium, pandas, pyarrow
print('mwparserfromhell', mwparserfromhell.__version__)
print('osmium OK | pandas', pandas.__version__, '| pyarrow', pyarrow.__version__)

## Block A — Cell 3: The checkpoint harness

One SQLite file on Drive holds: (a) per-source progress cursors, (b) a global seen-hash set for dedup, (c) buffered records flushed to parquet shards every `FLUSH_EVERY` records. All writes are transactional — a mid-write disconnect can't corrupt state.

In [ ]:
import os, datetime, subprocess, json, re
DUMP = f'{ROOT}/raw/enwikivoyage-latest-pages-articles.xml.bz2'

# --- dataset provenance: pin exactly which Wikivoyage dump this corpus used ---
st = os.stat(DUMP)
download_date = datetime.datetime.fromtimestamp(st.st_mtime).strftime('%Y-%m-%d')
gen = subprocess.run(f'bzcat "{DUMP}" | head -c 600', shell=True,
                     capture_output=True, text=True).stdout
generator = (re.search(r'<generator>([^<]+)</generator>', gen) or [None, '?'])[1]

# Wikimedia builds enwikivoyage pages-articles dumps on the 1st and 20th of each
# month and keeps dated copies for ~3 months. This corpus was built from the
# `latest` symlink, downloaded 2026-07-02 -> the 2026-07-01 (20260701) snapshot.
# The dated file has since expired from the server; provenance is fixed here.
PROVENANCE = {
    'source':          'English Wikivoyage XML dump (pages-articles)',
    'snapshot':        '20260701',
    'snapshot_note':   "resolved from the `latest` symlink; downloaded 2026-07-02; "
                       "Wikimedia 1st/20th dump cycle. Dated file expired from server.",
    'download_date':   download_date,
    'file':            os.path.basename(DUMP),
    'size_bytes':      st.st_size,
    'mediawiki_generator': generator,
    'url_pattern':     'https://dumps.wikimedia.org/enwikivoyage/20260701/'
                       'enwikivoyage-20260701-pages-articles.xml.bz2',
    'license':         'CC BY-SA 4.0',
    'shard_run':       'raw/ shards written 2026-07-26 22:33-23:14',
}
json.dump(PROVENANCE, open(f'{ROOT}/dataset_provenance.json', 'w'), indent=2)
for k, v in PROVENANCE.items():
    print(f'  {k:22}: {v}')
print(f'\nwrote {ROOT}/dataset_provenance.json  (cite this in the Dataset chapter)')


---

## K4 — Corpus composition table (for the thesis Dataset chapter)

Quantifies what the corpus actually *is*, rather than what it was designed to be. Three breakdowns
the write-up needs to state plainly:

1. **By source** — the project was scoped as Pakistan-focused; the Pakistan-specific pipelines
   (OSM PBF, Commons geosearch, `name:ur` gazetteer) contributed a tiny fraction. State the real
   split rather than implying a balanced multilingual corpus.
2. **By anchor quality** — `listing` records are individually geo-tagged; `page` records inherit
   one page-level `{{geo}}` centroid, so every sentence on a country or region article shares a
   single coarse coordinate. This is the mechanism behind the coarse geocells in Block E.
3. **By language routing** — how much non-English (and Urdu / Roman-Urdu) material actually exists.

Written to `results_i/corpus_composition.json` so the thesis table is generated, not retyped.


In [ ]:
import pandas as pd, glob, json, os, pyarrow.parquet as pq

os.makedirs(f'{ROOT}/results_i', exist_ok=True)
COLS = ['source', 'anchor_quality', 'listing_type', 'lat', 'place_name']

raw = pd.concat([pd.read_parquet(s, columns=[c for c in COLS
                 if c in pq.ParquetFile(s).schema.names])
                 for s in glob.glob(f'{ROOT}/raw/*_shard*.parquet')], ignore_index=True)
cv  = pd.read_parquet(f'{ROOT}/clean/corpus_v1.parquet',
                      columns=['source', 'anchor_quality', 'lat', 'place_name'])

def pct(n, d): return f'{n:,} ({n/d:.2%})'

print('=' * 74)
print('1. BY SOURCE')
print('=' * 74)
print(f'{"source":<16}{"raw records":>22}{"corpus_v1 (final)":>26}')
for s in sorted(set(raw.source) | set(cv.source)):
    print(f'{s:<16}{pct((raw.source==s).sum(), len(raw)):>22}'
          f'{pct((cv.source==s).sum(), len(cv)):>26}')
print(f'{"TOTAL":<16}{len(raw):>22,}{len(cv):>26,}')
pk_raw = raw.source.isin(['osm_pk', 'commons_pk']).sum()
print(f'\n-> Pakistan-specific sources: {pk_raw:,} of {len(raw):,} raw records '
      f'({pk_raw/len(raw):.2%}). The corpus is effectively global Wikivoyage.')

print('\n' + '=' * 74)
print('2. BY ANCHOR QUALITY  (how precisely each sentence is geo-located)')
print('=' * 74)
print(f'{"anchor_quality":<18}{"raw":>22}{"corpus_v1":>24}')
for a in sorted(set(raw.anchor_quality.dropna()) | set(cv.anchor_quality.dropna())):
    print(f'{a:<18}{pct((raw.anchor_quality==a).sum(), len(raw)):>22}'
          f'{pct((cv.anchor_quality==a).sum(), len(cv)):>24}')
page_frac = (cv.anchor_quality == 'page').mean()
print(f'\n-> {page_frac:.1%} of corpus_v1 is page-anchored: every sentence on an article '
      f'shares\n   that article\'s single {{{{geo}}}} centroid, so a country page contributes '
      f'dozens of\n   sentences at one coarse point. This is why Block E\'s geocells skew coarse.')

if 'listing_type' in raw:
    print('\n   raw records by listing template:')
    for t, n in raw.listing_type.value_counts().head(10).items():
        print(f'     {t:<14}{pct(n, len(raw))}')

print('\n' + '=' * 74)
print('3. BY LANGUAGE ROUTING  (Block C4)')
print('=' * 74)
stage_en = pq.ParquetFile(f'{ROOT}/clean/_stage_english.parquet').metadata.num_rows
stage_all = pq.ParquetFile(f'{ROOT}/clean/_stage_sentences.parquet').metadata.num_rows
urdu = pq.ParquetFile(f'{ROOT}/clean/held_urdu_bucket.parquet').metadata.num_rows
print(f'{"sentences after split (C3)":<40}{stage_all:>12,}')
print(f'{"routed English (C4)":<40}{stage_en:>12,}  ({stage_en/stage_all:.2%})')
print(f'{"held Urdu + Roman-Urdu (C4)":<40}{urdu:>12,}  ({urdu/stage_all:.4%})')
print(f'{"other languages (dropped)":<40}{stage_all-stage_en-urdu:>12,}')
print(f'\n-> {urdu} Urdu/Roman-Urdu sentences. The multilingual/Pakistan framing in the\n'
      f'   proposal is NOT supported by the collected data and must be scoped down.')

comp = {
 'raw_total': int(len(raw)), 'corpus_v1_total': int(len(cv)),
 'by_source_raw': {k: int(v) for k, v in raw.source.value_counts().items()},
 'by_source_corpus_v1': {k: int(v) for k, v in cv.source.value_counts().items()},
 'pakistan_specific_raw': int(pk_raw), 'pakistan_specific_frac': float(pk_raw/len(raw)),
 'by_anchor_quality_corpus_v1': {k: int(v) for k, v in cv.anchor_quality.value_counts().items()},
 'page_anchored_frac_corpus_v1': float(page_frac),
 'sentences_after_split': int(stage_all), 'routed_english': int(stage_en),
 'held_urdu_roman_urdu': int(urdu),
}
json.dump(comp, open(f'{ROOT}/results_i/corpus_composition.json', 'w'), indent=2)
print(f'\nwrote {ROOT}/results_i/corpus_composition.json')


In [ ]:
import sqlite3, hashlib, json, time, glob
import pandas as pd

STATE_DB = f'{ROOT}/checkpoints/state.db'
FLUSH_EVERY = 500

class Harness:
    """Checkpointed, deduplicating record sink. One instance per source."""
    def __init__(self, source):
        self.source = source
        self.buf = []
        self.db = sqlite3.connect(STATE_DB)
        self.db.execute('CREATE TABLE IF NOT EXISTS cursors '
                        '(source TEXT PRIMARY KEY, pos TEXT)')
        self.db.execute('CREATE TABLE IF NOT EXISTS seen '
                        '(hash TEXT PRIMARY KEY)')
        self.db.execute('CREATE TABLE IF NOT EXISTS shard_counter '
                        '(source TEXT PRIMARY KEY, n INTEGER)')
        self.db.commit()

    # ---- cursor (resume position) ----
    def get_cursor(self, default='0'):
        row = self.db.execute('SELECT pos FROM cursors WHERE source=?',
                              (self.source,)).fetchone()
        return row[0] if row else default

    def set_cursor(self, pos):
        self.db.execute('INSERT INTO cursors VALUES (?,?) '
                        'ON CONFLICT(source) DO UPDATE SET pos=excluded.pos',
                        (self.source, str(pos)))
        self.db.commit()

    # ---- dedup ----
    @staticmethod
    def h(text):
        return hashlib.sha1(text.strip().lower().encode()).hexdigest()

    def is_seen(self, text):
        return self.db.execute('SELECT 1 FROM seen WHERE hash=?',
                               (self.h(text),)).fetchone() is not None

    # ---- record sink ----
    def add(self, record):
        """record: dict with at least 'text'. Skips exact dupes."""
        text = record.get('text', '')
        if not text or self.is_seen(text):
            return False
        self.db.execute('INSERT OR IGNORE INTO seen VALUES (?)', (self.h(text),))
        record['source'] = self.source
        record['collected_at'] = time.strftime('%Y-%m-%dT%H:%M:%S')
        self.buf.append(record)
        if len(self.buf) >= FLUSH_EVERY:
            self.flush()
        return True

    def flush(self, cursor_pos=None):
        if self.buf:
            row = self.db.execute('SELECT n FROM shard_counter WHERE source=?',
                                  (self.source,)).fetchone()
            n = (row[0] if row else 0) + 1
            path = f'{ROOT}/raw/{self.source}_shard{n:05d}.parquet'
            pd.DataFrame(self.buf).to_parquet(path, index=False)
            self.db.execute('INSERT INTO shard_counter VALUES (?,?) '
                            'ON CONFLICT(source) DO UPDATE SET n=excluded.n',
                            (self.source, n))
            self.buf = []
        if cursor_pos is not None:
            self.set_cursor(cursor_pos)
        self.db.commit()

    def stats(self):
        shards = glob.glob(f'{ROOT}/raw/{self.source}_shard*.parquet')
        total = sum(len(pd.read_parquet(s)) for s in shards) if shards else 0
        return {'source': self.source, 'shards': len(shards),
                'records_on_disk': total, 'buffered': len(self.buf),
                'cursor': self.get_cursor()}

print('Harness ready. State DB:', STATE_DB)

## Block B1 — Cell 4: Download the Wikivoyage dump (~15 min on Colab's pipe)

One file, ~129 MB compressed. Saved to Drive so re-sessions skip the download.

**Pinned snapshot: `enwikivoyage-20260701`** (English Wikivoyage, MediaWiki 1.46.0-wmf.26).
This corpus was built from the `latest` symlink downloaded 2026-07-02; Wikimedia's dump cycle
(1st and 20th) makes that the 2026-07-01 snapshot. Dated dump files expire from the server after
~3 months — the provenance cell below records this permanently and writes `dataset_provenance.json`
for the Dataset chapter.

English Wikivoyage text is **CC BY-SA 4.0** — keep the `license` field downstream.


In [ ]:
DUMP = f'{ROOT}/raw/enwikivoyage-latest-pages-articles.xml.bz2'

# Pinned to the 2026-07-01 snapshot for reproducibility. If that dated file has
# expired from the server (Wikimedia keeps ~3 months), fall back to `latest` --
# a later snapshot changes the corpus, so record the new date if you re-run.
DUMP_SNAPSHOT = '20260701'
_dated  = (f'https://dumps.wikimedia.org/enwikivoyage/{DUMP_SNAPSHOT}/'
           f'enwikivoyage-{DUMP_SNAPSHOT}-pages-articles.xml.bz2')
_latest = ('https://dumps.wikimedia.org/enwikivoyage/latest/'
           'enwikivoyage-latest-pages-articles.xml.bz2')

if not os.path.exists(DUMP):
    rc = os.system(f'wget -q --show-progress -O "{DUMP}" "{_dated}"')
    if rc != 0 or os.path.getsize(DUMP) < 1_000_000:
        print(f'{DUMP_SNAPSHOT} unavailable, falling back to latest '
              f'(RECORD THE NEW DATE in dataset_provenance.json)')
        os.system(f'wget -q --show-progress -O "{DUMP}" "{_latest}"')

print('Dump size: %.1f MB' % (os.path.getsize(DUMP) / 1e6))


## Block B1 — Cell 5: The Wikivoyage listing parser

Streams the bz2 XML (never loads it fully into RAM — Colab free tier has ~12 GB and the decompressed dump is bigger). For every page, extracts:
- **Listing templates** `{{see|name=...|lat=...|long=...|content=...}}` and siblings → *pre-anchored records* (`anchor_quality='listing'`)
- **Page-level `{{geo|lat|long}}`** → fallback anchor for descriptive sentences in section prose (`anchor_quality='page'`)

Everything else is stripped. Resumes from the last page index if interrupted.

In [ ]:
import bz2, re
import xml.etree.ElementTree as ET
import mwparserfromhell as mwp
from tqdm.auto import tqdm

LISTING_TEMPLATES = {'see', 'do', 'eat', 'drink', 'sleep', 'buy', 'listing',
                     'marker'}
# English Wikivoyage text migrated to CC BY-SA 4.0 (Wikimedia relicensing, 2023).
# Records already written to raw/ carry the old '3.0' string -- it is provenance
# metadata only; the authoritative licence is in dataset_provenance.json.
WV_LICENSE = 'CC-BY-SA-4.0'

def _clean(wikicode_or_str):
    """Wiki markup -> plain text."""
    try:
        code = (wikicode_or_str if hasattr(wikicode_or_str, 'strip_code')
                else mwp.parse(str(wikicode_or_str)))
        txt = code.strip_code(normalize=True, collapse=True)
    except Exception:
        txt = str(wikicode_or_str)
    txt = re.sub(r'https?://\S+', '', txt)
    return re.sub(r'\s+', ' ', txt).strip()

def _get(tpl, *names):
    for n in names:
        if tpl.has(n):
            v = _clean(tpl.get(n).value)
            if v:
                return v
    return None

def _coord(v, kind='lon'):
    """Parse a coordinate string. kind='lat' bounds at +-90, 'lon' at +-180.
    ponytail: earlier version bounded everything at +-180, so a swapped
              lat/lon passed. Pass kind explicitly at each call site."""
    try:
        f = float(v)
    except (TypeError, ValueError):
        return None
    lim = 90.0 if kind == 'lat' else 180.0
    return f if -lim <= f <= lim else None

def parse_page(title, wikitext, harness):
    """Extract anchored records from one Wikivoyage page. Returns count."""
    added = 0
    try:
        code = mwp.parse(wikitext)
    except Exception:
        return 0

    # page-level geo anchor (fallback for prose)
    page_lat = page_lon = None
    for tpl in code.filter_templates():
        if tpl.name.strip().lower() == 'geo' and len(tpl.params) >= 2:
            page_lat = _coord(tpl.params[0].value.strip_code(), 'lat')
            page_lon = _coord(tpl.params[1].value.strip_code(), 'lon')
            break

    # 1) listing templates: individually anchored, highest value
    for tpl in code.filter_templates():
        tname = tpl.name.strip().lower()
        if tname not in LISTING_TEMPLATES:
            continue
        lat = _coord(_get(tpl, 'lat', 'latitude'), 'lat')
        lon = _coord(_get(tpl, 'long', 'lon', 'longitude'), 'lon')
        name = _get(tpl, 'name')
        content = _get(tpl, 'content', 'description')
        if content and len(content) > 25:
            added += harness.add({
                'text': content, 'place_name': name, 'page': title,
                'lat': lat if lat is not None else page_lat,
                'lon': lon if lon is not None else page_lon,
                'anchor_quality': 'listing' if lat is not None else 'page',
                'listing_type': tname, 'lang': 'en',
                'license': WV_LICENSE,
            })

    # 2) descriptive prose in Understand/See/Districts sections, page-anchored
    if page_lat is not None:
        prose = _clean(code)
        for sent in re.split(r'(?<=[.!?])\s+', prose):
            if 60 < len(sent) < 400 and not sent.startswith(('=', '*', '#')):
                added += harness.add({
                    'text': sent, 'place_name': title, 'page': title,
                    'lat': page_lat, 'lon': page_lon,
                    'anchor_quality': 'page', 'listing_type': 'prose',
                    'lang': 'en', 'license': WV_LICENSE,
                })
    return added

def run_wikivoyage(dump_path, limit_pages=None):
    hv = Harness('wikivoyage')
    start_at = int(hv.get_cursor('0'))
    print(f'Resuming from page index {start_at}')
    NS = None  # derived from the first element -- dump-version independent
    idx = added = 0
    try:
        with bz2.open(dump_path, 'rb') as f:
            for _, elem in tqdm(ET.iterparse(f, events=('end',)), unit=' elems'):
                if NS is None and elem.tag.startswith('{'):
                    NS = elem.tag.split('}')[0] + '}'
                if not elem.tag.endswith('page'):
                    continue
                idx += 1
                if idx <= start_at:
                    elem.clear(); continue
                ns_el = elem.find(f'{NS}ns')
                if ns_el is not None and ns_el.text == '0':   # articles only
                    title_el = elem.find(f'{NS}title')
                    text_el = elem.find(f'{NS}revision/{NS}text')
                    if title_el is not None and text_el is not None and text_el.text:
                        added += parse_page(title_el.text, text_el.text, hv)
                elem.clear()
                if idx % 200 == 0:
                    hv.flush(cursor_pos=idx)   # checkpoint
                if limit_pages and (idx - start_at) >= limit_pages:
                    break
    except KeyboardInterrupt:
        print('Interrupted — state saved, just re-run this cell to resume.')
    finally:
        hv.flush(cursor_pos=idx)
    print(f'Pages scanned this run: {idx - start_at}, records added: {added}')
    print(hv.stats())

# Smoke test on 300 pages first — expect a few dozen records:
run_wikivoyage(DUMP, limit_pages=300)


## Block B1 — Cell 6: Full run

~60k pages, expect **1.5–3 hours** on Colab CPU and on the order of **100k+ listing records** globally. It checkpoints every 200 pages — you can interrupt, close, reopen, re-run this cell, and it continues. Keep the tab open.

In [ ]:
run_wikivoyage(DUMP)   # full pass; safe to interrupt & re-run

## Block B2 — Cell 7: OSM Pakistan — descriptions + gazetteer (~30–45 min)

One pyosmium pass produces **two outputs**: (1) anchored description texts, (2) the name gazetteer (`name`, `name:ur`, `alt_name` → id/lat/lon) you'll need for toponym resolution in v2 — free to grab now. OSM data: ODbL — keep derived fields separable.

In [ ]:
PBF = f'{ROOT}/raw/pakistan-latest.osm.pbf'
if not os.path.exists(PBF):
    !wget -q --show-progress -O "$PBF" \
      https://download.geofabrik.de/asia/pakistan-latest.osm.pbf
print('PBF size: %.1f MB' % (os.path.getsize(PBF)/1e6))

In [ ]:
import osmium

TEXT_TAGS = ('description', 'description:en', 'note', 'inscription')
NAME_TAGS = ('name', 'name:en', 'name:ur', 'alt_name', 'old_name')

class OSMHandler(osmium.SimpleHandler):
    def __init__(self, harness):
        super().__init__()
        self.hv = harness
        self.gaz = []          # gazetteer rows
        self.n_text = 0

    def _process(self, o, lat, lon, otype):
        tags = dict(o.tags)
        name = tags.get('name')
        # gazetteer row for anything named with a location
        if name and lat is not None:
            for nt in NAME_TAGS:
                if nt in tags:
                    self.gaz.append({'variant': tags[nt], 'canonical': name,
                                     'osm_type': otype, 'osm_id': o.id,
                                     'lat': lat, 'lon': lon,
                                     'fclass': tags.get('place') or
                                               tags.get('amenity') or
                                               tags.get('tourism') or
                                               tags.get('historic') or ''})
        # description text record
        for tt in TEXT_TAGS:
            txt = tags.get(tt, '')
            if len(txt) > 25 and lat is not None:
                self.n_text += self.hv.add({
                    'text': txt, 'place_name': name, 'page': None,
                    'lat': lat, 'lon': lon, 'anchor_quality': 'element',
                    'listing_type': f'osm_{tt}', 'lang': 'en',
                    'license': 'ODbL', 'osm_type': otype, 'osm_id': o.id,
                })

    def node(self, n):
        if n.location.valid():
            self._process(n, n.location.lat, n.location.lon, 'node')

    def way(self, w):
        # centroid-ish: first node location if resolved; keep it cheap
        try:
            loc = w.nodes[0].location
            lat, lon = (loc.lat, loc.lon) if loc.valid() else (None, None)
        except Exception:
            lat = lon = None
        self._process(w, lat, lon, 'way')

hv_osm = Harness('osm_pk')
handler = OSMHandler(hv_osm)
handler.apply_file(PBF, locations=True, idx='flex_mem')
hv_osm.flush(cursor_pos='done')

gaz_df = pd.DataFrame(handler.gaz).drop_duplicates(subset=['variant', 'osm_id'])
gaz_df.to_parquet(f'{ROOT}/clean/gazetteer_pk.parquet', index=False)
print(f'Description records: {handler.n_text} | gazetteer rows: {len(gaz_df)}')
print(hv_osm.stats())

## Block B3 — Cell 8: Wikimedia Commons geotagged captions (grid sweep)

Sweeps a lat/lon grid over Pakistan with the `geosearch` API (10 km radius per tile), pulls `ImageDescription` for geotagged files. Rate-limited politely; checkpoints per tile so it resumes mid-grid. Run this after B2 finishes, or in your second parallel notebook.

In [ ]:
import requests, itertools, html

API = 'https://commons.wikimedia.org/w/api.php'
HEADERS = {'User-Agent': 'geospatial-nlp-research/0.1 (academic project)'}

# Pakistan bounding box, 0.25-degree grid (~2200 tiles)
LATS = [round(23.5 + 0.25*i, 2) for i in range(int((37.5-23.5)/0.25))]
LONS = [round(60.5 + 0.25*i, 2) for i in range(int((77.5-60.5)/0.25))]
TILES = list(itertools.product(LATS, LONS))

TAG_RE = re.compile(r'<[^>]+>')

def commons_tile(lat, lon, session):
    r = session.get(API, params={
        'action': 'query', 'format': 'json', 'generator': 'geosearch',
        'ggscoord': f'{lat}|{lon}', 'ggsradius': 10000, 'ggslimit': 50,
        'ggsnamespace': 6,  # File:
        'prop': 'imageinfo|coordinates', 'iiprop': 'extmetadata',
        'iiextmetadatafilter': 'ImageDescription|LicenseShortName',
    }, headers=HEADERS, timeout=30)
    out = []
    for p in (r.json().get('query', {}) or {}).get('pages', {}).values():
        try:
            md = p['imageinfo'][0]['extmetadata']
            desc = TAG_RE.sub(' ', html.unescape(
                md.get('ImageDescription', {}).get('value', '')))
            desc = re.sub(r'\s+', ' ', desc).strip()
            co = p.get('coordinates', [{}])[0]
            if len(desc) > 25 and 'lat' in co:
                out.append({'text': desc, 'place_name': p.get('title'),
                            'page': p.get('title'),
                            'lat': co['lat'], 'lon': co['lon'],
                            'anchor_quality': 'element',
                            'listing_type': 'commons_caption', 'lang': 'unk',
                            'license': md.get('LicenseShortName', {})
                                         .get('value', 'unknown')})
        except (KeyError, IndexError):
            continue
    return out

def run_commons():
    hv = Harness('commons_pk')
    start = int(hv.get_cursor('0'))
    print(f'Resuming at tile {start}/{len(TILES)}')
    s = requests.Session()
    for i in tqdm(range(start, len(TILES)), initial=start, total=len(TILES)):
        lat, lon = TILES[i]
        try:
            for rec in commons_tile(lat, lon, s):
                hv.add(rec)
        except requests.RequestException:
            time.sleep(5)      # transient error: brief backoff, keep going
        if i % 50 == 0:
            hv.flush(cursor_pos=i)
        time.sleep(0.6)        # ~1.5 req/s, polite
    hv.flush(cursor_pos=len(TILES))
    print(hv.stats())

run_commons()

## Cell 9: Collection dashboard — run anytime

In [ ]:
import glob
rows = []
for src in ['wikivoyage', 'osm_pk', 'commons_pk']:
    shards = glob.glob(f'{ROOT}/raw/{src}_shard*.parquet')
    n = sum(len(pd.read_parquet(s, columns=['text'])) for s in shards)
    rows.append({'source': src, 'shards': len(shards), 'records': n})
df = pd.DataFrame(rows)
df.to_csv(f'{ROOT}/logs/collection_status.csv', index=False)
print(df.to_string(index=False))
print('\nTotal:', df.records.sum(), 'records ->', f'{ROOT}/raw/')